In [1]:
!pip install -qU langchain-groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 5.5 MB/s eta 0:00:00


In [2]:
!pip install -qU langchain-core

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 557.4/557.4 kB 11.4 MB/s eta 0:00:00


In [3]:
!pip install -qU langfuse

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 607.4/607.4 kB 4.9 MB/s eta 0:00:00


In [4]:
!pip install -qU langfuse langchain-openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 120.4/120.4 kB 2.7 MB/s eta 0:00:00


In [5]:
import os
from google.colab import userdata

groq = userdata.get("Expense_groq")
os.environ["GROQ_API_KEY"] = groq

In [6]:
Langfuse_S_key = userdata.get("adv_lf")
Langfuse_P_key= userdata.get("adv_lfp")

In [7]:
os.environ["LANGFUSE_SECRET_KEY"] = Langfuse_S_key
os.environ["LANGFUSE_PUBLIC_KEY"] = Langfuse_P_key
os.environ["LANGFUSE_BASE_URL"] = "https://us.cloud.langfuse.com"

In [8]:
#LANGFUSE_SECRET_KEY="sk-lf-aac54b0d-520c-4208-8683-6d346d761c84"
#LANGFUSE_PUBLIC_KEY="pk-lf-6ff39959-fa7c-49b4-894c-154fbd26fed3"
#LANGFUSE_BASE_URL="https://us.cloud.langfuse.com"

In [9]:
from langfuse.langchain import CallbackHandler

langfuse_handler = CallbackHandler()

In [10]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="qwen/qwen3-32b",
    reasoning_format="hidden"
)

In [11]:
llm

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.4.8', 'langchain': '1.3.9'}}, profile={'name': 'Qwen3 32B', 'release_date': '2024-12-23', 'last_updated': '2024-12-23', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 40960, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x7f97dc9a30e0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x7f97dc7f83e0>, model_name='qwen/qwen3-32b', reasoning_format='hidden', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [12]:
!pip install -qU langchain-core

In [13]:
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field

In [14]:

class ExpenseCategoryOutput(BaseModel):
    expense_category: str = Field(
        description="Expense category"
    )

class ExpenseTypeOutput(BaseModel):
    expense_type: str = Field(
        description="Essential, Non-Essential, Uncertain"
    )

class ConfidenceOutput(BaseModel):
    confidence_note: str = Field(
        description="Reasoning note"
    )

In [15]:
from langchain_core.output_parsers import PydanticOutputParser

category_parser = PydanticOutputParser(
    pydantic_object=ExpenseCategoryOutput
)

type_parser = PydanticOutputParser(
    pydantic_object=ExpenseTypeOutput
)

confidence_parser = PydanticOutputParser(
    pydantic_object=ConfidenceOutput
)

In [16]:
from langchain_core.prompts import ChatPromptTemplate

category_prompt = ChatPromptTemplate.from_template(
"""
Classify the expense into ONE category.

Categories:
- Food & Groceries
- Transportation
- Utilities
- Housing
- Healthcare
- Entertainment
- Shopping
- Education
- Travel
- Miscellaneous

Expense:
{expense_text}

{format_instructions}
""",
partial_variables={
    "format_instructions":
    category_parser.get_format_instructions()
}
)

In [17]:
from langchain_core.runnables import RunnableLambda

In [18]:
category_chain = (
    category_prompt
    | llm
    | category_parser
    | RunnableLambda(
        lambda x: x.expense_category
    )
)

In [19]:
type_prompt = ChatPromptTemplate.from_template(
"""
Determine spending type.

Labels:
- Essential
- Non-Essential
- Uncertain

Rules:
Essential:
Food, rent, utility bills, fuel,
medical expenses, education.

Non-Essential:
Luxury items, entertainment,
premium subscriptions, gadgets.

Uncertain:
Not enough information.

Expense:
{expense_text}

{format_instructions}
""",
partial_variables={
    "format_instructions":
    type_parser.get_format_instructions()
}
)

In [20]:
type_chain = (
    type_prompt
    | llm
    | type_parser
    | RunnableLambda(
        lambda x: x.expense_type
    )
)

In [21]:
confidence_prompt = ChatPromptTemplate.from_template(
"""
Generate a short confidence note.

Rules:
1. Explain classification briefly.
2. Maximum one sentence.
3. If description is ambiguous,
   mention uncertainty.

Expense:
{expense_text}

{format_instructions}
""",
partial_variables={
    "format_instructions":
    confidence_parser.get_format_instructions()
}
)

In [22]:
confidence_chain = (
    confidence_prompt
    | llm
    | confidence_parser
    | RunnableLambda(
        lambda x: x.confidence_note
    )
)

In [23]:
from langchain_core.runnables import RunnableParallel

expense_analysis = RunnableParallel({"expense_category": category_chain,
                                     "expense_type": type_chain,
                                     "confidence_note": confidence_chain})

In [24]:
expense_text1= """Paid monthly electricity bill"""
expense_text2 = """Bought groceries from supermarket"""
expense_text3= """Bought premium smartwatch"""

In [25]:
result = expense_analysis.invoke(
{"expense_text": expense_text1},
config={
        "callbacks": [langfuse_handler]
        })

In [26]:
result

{'expense_category': 'Utilities',
 'expense_type': 'Essential',
 'confidence_note': "The expense 'Paid monthly electricity bill' is classified as a utility expense with clear description."}

In [27]:

result1 = expense_analysis.invoke(
{ "expense_text": expense_text2},
config={
    "callbacks": [langfuse_handler]
}
)

In [28]:
result1

{'expense_category': 'Food & Groceries',
 'expense_type': 'Essential',
 'confidence_note': "Classified as 'Groceries' with high confidence due to explicit mention of 'supermarket'."}

In [29]:

result3 = expense_analysis.invoke(
{"expense_text": expense_text3},
config={
    "callbacks": [langfuse_handler]
}
)


In [30]:
result3

{'expense_category': 'Shopping',
 'expense_type': 'Non-Essential',
 'confidence_note': 'Expense classified as electronics.'}

In [ ]:
#Checks whether confidence note is meaningful